# Consolidated_layer

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col, count, when

df_silver = spark.table(
    "workspace.transformation_silver.silver_flights_clean"
)

#### gold_flight_summary

In [0]:
df_airline = df_silver.withColumn(
    "airline_code",
    substring("callsign", 1, 3)
)

In [0]:
df_flight_summary = df_airline.groupBy(
    "airline_code"
).agg(

    count("*").alias("total_flights"),

    avg("velocity_kmh").alias("avg_speed_kmh"),

    sum(
        when(col("on_ground") == True, 1).otherwise(0)
    ).alias("on_ground_flights"),

    sum(
        when(col("on_ground") == False, 1).otherwise(0)
    ).alias("airborne_flights")
)

In [0]:
df_flight_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.aggregation_gold.gold_flight_summary"
    )

#### gold_country_agg

In [0]:
df_country_agg = df_silver.groupBy(
    "origin_country",
    "country_flag"
).agg(

    count("*").alias("total_flights"),

    avg("velocity_kmh").alias("avg_speed_kmh"),

    sum(
        when(col("on_ground") == True, 1).otherwise(0)
    ).alias("on_ground_flights"),

    sum(
        when(col("on_ground") == False, 1).otherwise(0)
    ).alias("airborne_flights")
)

In [0]:
display(df_country_agg)

origin_country,country_flag,total_flights,avg_speed_kmh,on_ground_flights,airborne_flights
India,IN,168,640.0144285714289,15,153
United States,US,1953,585.0220758196716,167,1786
Chile,OTHER,28,641.709,2,26
Hungary,OTHER,52,674.5146923076925,3,49
Turkey,OTHER,233,743.8479141630895,8,225
Sweden,OTHER,86,614.545953488372,7,79
Indonesia,OTHER,31,702.4621935483871,0,31
Bulgaria,OTHER,12,655.155,2,10
Austria,OTHER,149,543.6606442953018,12,137
Germany,DE,398,481.7487437185934,37,361


In [0]:
df_country_agg.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.aggregation_gold.gold_country_agg"
    )

#### gold_hourly_volume

In [0]:
df_hourly = df_silver.withColumn(
    "hour",
    hour(col("timestamp"))
)

In [0]:
df_hourly_volume = df_hourly.groupBy(
    "hour"
).agg(
    count("*").alias("flight_volume")
)

In [0]:
df_hourly_volume.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.aggregation_gold.gold_hourly_volume"
    )

#### gold_speed_dist

In [0]:
df_speed_band = df_silver.withColumn(
    "speed_band",

    when(col("velocity_kmh") < 200, "<200")

    .when(
        (col("velocity_kmh") >= 200) &
        (col("velocity_kmh") < 500),
        "200-500"
    )

    .when(
        (col("velocity_kmh") >= 500) &
        (col("velocity_kmh") < 800),
        "500-800"
    )

    .otherwise("800+")
)

In [0]:
df_speed_dist = df_speed_band.groupBy(
    "speed_band"
).count()

In [0]:
df_speed_dist.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.aggregation_gold.gold_speed_dist"
    )

#### gold_pipeline_health

In [0]:
df_pipeline_health = df_silver.groupBy(
    "sla_status"
).agg(

    count("*").alias("record_count"),

    avg("processing_lag_seconds")
    .alias("avg_lag_seconds")
)

In [0]:
df_pipeline_health.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.aggregation_gold.gold_pipeline_health"
    )